In [1]:
import json
import os
import time
from tqdm import tqdm
from openai import OpenAI
from dotenv import load_dotenv

# ==========================================
# 1. SETUP API & PATHS
# ==========================================
load_dotenv()
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

input_file = "/workspaces/Arch_PC_Assistent/outputs/benchmark/benchmark_results.json"
output_file = "/workspaces/Arch_PC_Assistent/outputs/benchmark/final_scores.json"

# ==========================================
# 2. THE JUDGE PROMPT
# ==========================================
def evaluate_answer(question, context, answer):
    system_prompt = """You are an expert impartial judge evaluating an AI assistant's response to an Arch Linux / Hyprland user.
You will be provided with the User's Question, the Retrieved RAG Context, and the AI's Answer.

Evaluate the answer based on three criteria, scoring each from 0 to 10:
1. "format_score": Does the output STRICTLY follow the `<think>...</think><answer>...</answer>` format? (0 if missing, 5 if malformed, 10 if perfect).
2. "accuracy_score": Is the technical advice correct and safe? Did it properly use the RAG context without hallucinating?
3. "helpfulness_score": Is the answer direct, easy to follow, and does it solve the user's core problem?

Respond ONLY with a valid JSON object in this exact format:
{
    "format_score": 10,
    "accuracy_score": 9,
    "helpfulness_score": 8,
    "critique": "A short 1-sentence explanation of your scores."
}"""

    user_prompt = f"""QUESTION: {question}

RETRIEVED CONTEXT:
{context}

AI ANSWER TO EVALUATE:
{answer}"""

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0, # Temperatur 0 für maximale Objektivität und Konsistenz
            response_format={"type": "json_object"} # Erzwingt sauberes JSON
        )
        
        result_json = json.loads(response.choices[0].message.content.strip())
        return result_json
        
    except Exception as e:
        print(f"Error during API call: {e}")
        return {"format_score": 0, "accuracy_score": 0, "helpfulness_score": 0, "critique": "API Error."}

# ==========================================
# 3. EVALUATION LOOP
# ==========================================
def run_evaluation():
    with open(input_file, "r", encoding="utf-8") as f:
        data = json.load(f)
        
    print(f"Starting evaluation of {len(data)} questions (x3 models = {len(data)*3} API calls)...")
    
    # Tracking total scores
    totals = {
        "base": {"format": 0, "accuracy": 0, "helpfulness": 0},
        "sft": {"format": 0, "accuracy": 0, "helpfulness": 0},
        "rsft": {"format": 0, "accuracy": 0, "helpfulness": 0}
    }
    
    for i, item in enumerate(tqdm(data, desc="Judging answers")):
        item["scores"] = {}
        
        for model_key in ["base", "sft", "rsft"]:
            # Evaluate
            score_data = evaluate_answer(item["question"], item["context"], item["answers"][model_key])
            item["scores"][model_key] = score_data
            
            # Add to totals
            totals[model_key]["format"] += score_data.get("format_score", 0)
            totals[model_key]["accuracy"] += score_data.get("accuracy_score", 0)
            totals[model_key]["helpfulness"] += score_data.get("helpfulness_score", 0)
            
            # Kurze Pause, um API-Ratelimits zu vermeiden
            time.sleep(0.5)

    # ==========================================
    # 4. PRINT FINAL REPORT
    # ==========================================
    num_q = len(data)
    print("\n" + "="*50)
    print("🏆 FINAL BENCHMARK RESULTS 🏆")
    print("="*50)
    
    for model_key in ["base", "sft", "rsft"]:
        avg_fmt = totals[model_key]["format"] / num_q
        avg_acc = totals[model_key]["accuracy"] / num_q
        avg_hlp = totals[model_key]["helpfulness"] / num_q
        total_avg = (avg_fmt + avg_acc + avg_hlp) / 3
        
        print(f"\nModel: [{model_key.upper()}]")
        print(f"  - Format Adherence: {avg_fmt:.1f} / 10.0")
        print(f"  - Technical Accuracy: {avg_acc:.1f} / 10.0")
        print(f"  - Helpfulness:      {avg_hlp:.1f} / 10.0")
        print(f"  => OVERALL SCORE:   {total_avg:.1f} / 10.0")
        
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
        
    print(f"\nDetailed evaluation saved to {output_file}")

if __name__ == "__main__":
    run_evaluation()

Starting evaluation of 50 questions (x3 models = 150 API calls)...


Judging answers: 100%|██████████| 50/50 [06:10<00:00,  7.40s/it]


🏆 FINAL BENCHMARK RESULTS 🏆

Model: [BASE]
  - Format Adherence: 9.7 / 10.0
  - Technical Accuracy: 4.3 / 10.0
  - Helpfulness:      5.0 / 10.0
  => OVERALL SCORE:   6.3 / 10.0

Model: [SFT]
  - Format Adherence: 8.2 / 10.0
  - Technical Accuracy: 5.3 / 10.0
  - Helpfulness:      5.8 / 10.0
  => OVERALL SCORE:   6.4 / 10.0

Model: [RSFT]
  - Format Adherence: 9.4 / 10.0
  - Technical Accuracy: 5.8 / 10.0
  - Helpfulness:      6.2 / 10.0
  => OVERALL SCORE:   7.1 / 10.0

Detailed evaluation saved to /workspaces/Arch_PC_Assistent/outputs/benchmark/final_scores.json
